In [1]:
import cosima_cookbook as cc
from cosima_cookbook import distributed as ccd
import matplotlib.pyplot as plt
import numpy as np
import netCDF4 as nc
import xarray as xr
import glob,os
import cmocean.cm as cmocean

import logging
logging.captureWarnings(True)
logging.getLogger('py.warnings').setLevel(logging.ERROR)

from dask.distributed import Client

In [2]:
client = Client(memory_limit='500gb')
client

2023-12-28 13:53:26,525 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 500gb due to system memory limit of 125.20 GiB
2023-12-28 13:53:26,533 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 500gb due to system memory limit of 125.20 GiB
2023-12-28 13:53:26,538 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 500gb due to system memory limit of 125.20 GiB
2023-12-28 13:53:26,543 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 500gb due to system memory limit of 125.20 GiB
2023-12-28 13:53:26,549 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 500gb due to system memory limit of 125.20 GiB
2023-12-28 13:53:26,554 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 500gb due to system memory limit of 125.20 GiB
2023-12-28 13:53:26,559 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 500gb due to system memory limit of 125.20 GiB


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 7
Total threads: 14,Total memory: 876.39 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33299,Workers: 7
Dashboard: http://127.0.0.1:8787/status,Total threads: 14
Started: Just now,Total memory: 876.39 GiB
Comm: tcp://127.0.0.1:44367,Total threads: 2
Dashboard: http://127.0.0.1:44493/status,Memory: 125.20 GiB
Nanny: tcp://127.0.0.1:40207,


In [3]:
	monthdays = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
	month = '1'
	month = month.zfill(2)
	year = str(2000)
	start_time=year+'-'+month  
	 #Start_time0 and end_time0 are for importing the daily transport, and it ahsthe number of days in the month    
	start_time0=year+'-'+month +'-01'     
	end_time0=year+'-'+month +'-' + str(monthdays[int(int(1)-1)])
	print(start_time0) 
	print(end_time0) 
	exp = 'panant-005-zstar-ACCESSyr2'
	
	print("Start date =" + start_time) 
	year2=str(int(start_time[0:4])+1)
	month2=str(int(start_time[5:7])+1)
	month2=str(int(month2))
	month2 = month2.zfill(2)    
	print("month2 is =" + month2) 
	print("year2 is =" + str(year2))     
	
	imon = int(1)
	if imon <12:
		end_time=year+'-'+month2
	else:
		end_time=year2+'-01'
	
	print("End date =" + end_time) 
	
	
	time_period = str(int(start_time[:4]))+'-'+str(int(end_time[:4]))
	

2000-01-01
2000-01-31
Start date =2000-01
month2 is =02
year2 is =2001
End date =2000-02


In [4]:
	session = cc.database.create_session()
	rho_0 = 1035.0

	lat_range = slice(-90,-59)
	
	isobath_depth = 1000
	
	
	
	print("importing isobath mask")
	#outfile = '/g/data/v45/akm157/model_data/access-om2/Antarctic_slope_contour_'+str(isobath_depth)+'m.npz'
	outfile = '/home/156/wf4500/v45_wf4500/Project_panan/GH/Panan_HT_ASC/contours/Antarctic_slope_contour_Panan005_'+str(isobath_depth)+'m.npz'
	data = np.load(outfile)
	mask_y_transport = data['mask_y_transport']
	mask_x_transport = data['mask_x_transport']
	mask_y_transport_numbered = data['mask_y_transport_numbered']
	mask_x_transport_numbered = data['mask_x_transport_numbered']
	
	yh = cc.querying.getvar(exp,'yh',session,n=1)
	yh = yh.sel(yh=lat_range)
	yq = cc.querying.getvar(exp,'yq',session,n=1)
	yq = yq.sel(yq=lat_range)
	xh = cc.querying.getvar(exp,'xh',session,n=1)
	xq = cc.querying.getvar(exp,'xq',session,n=1)
	xq=xq[:-1]; #Added for panan to cutout the extra xlon in this grid
	
	mask_x_transport =xr.DataArray(data['mask_x_transport']).assign_coords({"dim_0": np.array(yh),"dim_1": np.array(xq)}).rename(dim_0="yh",dim_1="xq")
	mask_y_transport =xr.DataArray(data['mask_y_transport']).assign_coords({"dim_0": np.array(yh),"dim_1": np.array(xq)}).rename(dim_0="yq",dim_1="xh")
	mask_x_transport_numbered =xr.DataArray(data['mask_x_transport_numbered']).assign_coords({"dim_0": np.array(yh),"dim_1": np.array(xh)}).rename(dim_0="yh",dim_1="xh")
	mask_y_transport_numbered =xr.DataArray(data['mask_y_transport_numbered']).assign_coords({"dim_0": np.array(yh),"dim_1": np.array(xh)}).rename(dim_0="yh",dim_1="xh")
	
	num_points = int(np.maximum(np.max(mask_y_transport_numbered),np.max(mask_x_transport_numbered)))
	lat_along_contour = np.zeros((num_points))
	lon_along_contour = np.zeros((num_points))
	
	# locations for zonal transport:
	x_indices_masked = mask_x_transport_numbered.stack().values
	x_indices = np.sort(x_indices_masked[x_indices_masked>0])
	for count in x_indices:
		count = int(count)
		jj = int(np.where(mask_x_transport_numbered==count)[0])
		ii = int(np.where(mask_x_transport_numbered==count)[1])   
		lon_along_contour[count-1] = xq[ii].values
		lat_along_contour[count-1] = mask_x_transport_numbered.yh[jj].values
	
	# locations for meridional transport:
	y_indices_masked = mask_y_transport_numbered.stack().values
	y_indices = np.sort(y_indices_masked[y_indices_masked>0])
	for count in y_indices:
		count = int(count)
		jj = np.where(mask_y_transport_numbered==count)[0]
		ii = np.where(mask_y_transport_numbered==count)[1]
		lon_along_contour[count-1] = mask_x_transport_numbered.xh[ii].values
		lat_along_contour[count-1] = yq[jj].values
	


importing isobath mask
0.3.0
0.3.0
0.3.0
0.3.0


In [5]:
	print("importing daily mass transports and salinities")
	depth_sliceint=slice(0,51)
	
	
	#importing all heat transports calculated offline
	Sal_daily = cc.querying.getvar(exp,'so',session,ncfile='%daily%',start_time=start_time,end_time=end_time,chunks={'xh':'100gb','yh':'100gb'}).sel(time=slice(start_time,end_time))

	Sal_daily = Sal_daily.sel(yh=lat_range).isel(z_l=depth_sliceint).rename({'z_l':'z_l_sub01'})
	
	
	# save a long term average of vhrho_nt and uhrho_et:
	# outpath = '/g/data/x77/wf4500/ASC_project/uhrho_vhrho_'+start_time+'.nc'
	# check if already exists:
	vhrho_nt = cc.querying.getvar(exp,'vo',session,ncfile='%daily%',start_time=start_time,end_time=end_time,chunks={'xh':'100gb','yq':'100gb'}).sel(time=slice(start_time,end_time))
	uhrho_et = cc.querying.getvar(exp,'uo',session,ncfile='%daily%',start_time=start_time,end_time=end_time,chunks={'xq':'100gb','yh':'100gb'}).sel(time=slice(start_time,end_time))
	
	vhrho_nt = vhrho_nt.sel(yq=lat_range).sel(time=slice(start_time0,end_time0))
	uhrho_et = uhrho_et.sel(yh=lat_range).sel(time=slice(start_time0,end_time0))
	
	vhrho_nttime=vhrho_nt.time
	ndays_month=int(np.size(vhrho_nt.time));
	
	vol = cc.querying.getvar(exp,'volcello',session,ncfile='%daily_z%',start_time=start_time,end_time=end_time,chunks={'xh':'100gb','yh':'100gb'}).sel(time=slice(start_time,end_time))
	vol = vol.isel(z_l=depth_sliceint).rename({'z_l':'z_l_sub01'})
	area = cc.querying.getvar(exp,'areacello',session,ncfile='%static%',n=1,chunks={'xh':'100gb','yh':'100gb'})
	Dzt = vol/area
	del vol,area
	# outpath = '/g/data/x77/wf4500/ASC_project/model_data/access-om2/'+exp+'/Antarctic_cross_slope/Daily/uhrho_vhrho_'+start_time+'.nc'
	ds = xr.Dataset({'vhrho_nt': vhrho_nt,'uhrho_et':uhrho_et})
	#del ds.vhrho_nt.attrs['time_bounds']
	#del ds.uhrho_et.attrs['time_bounds']
	#ds.to_netcdf(outpath)
	#ds.close()
	# print('Daily data being calculated for month =  ...')
	# print(vhrho_nttime)
	
	import os
	#outpath = '/g/data/x77/wf4500/ASC_project/model_data/access-om2/'+exp+'/Antarctic_cross_slope/Daily/uhrho_vhrho_'+start_time+'.nc'
	#ds = xr.open_dataset(outpath)
	vhrho_nt = ds['vhrho_nt']
	uhrho_et = ds['uhrho_et']
	
	# subtract freezing point heat transport:
	yh = cc.querying.getvar(exp,'yh',session,n=1)
	dxu = cc.querying.getvar(exp,'dxCv',session,n=1) #on OM2 is dxu
	dyt = cc.querying.getvar(exp,'dyCu',session,n=1)# on OM2 is dyu
	
	## give dxu and dyt correct coordinates:
	## dxu.coords['nj'] = yh.values
	## dxu.coords['ni'] = xh['xh'].values
	## dxu = dxu.rename(({'ni':'xh', 'nj':'yh'}))
	## dyt.coords['nj'] = yh.values
	## dyt.coords['ni'] = xh['xh'].values
	## dyt = dyt.rename(({'ni':'xh', 'nj':'yh'}))
	## # select latitude range:
	dxu = dxu.sel(yq=lat_range)
	dyt = dyt.sel(yh=lat_range)
	
	
	
	## # Note vhrho_nt is v*dz*1035 and is positioned on north centre edge of t-cell.
	## # sum in depth:
	## vhrho_nt = vhrho_nt
	## uhrho_et = uhrho_et
	## # convert to transport:
	vhrho_nt = vhrho_nt*dxu#*rho_0
	uhrho_et = uhrho_et*dyt#*rho_0
	
	# # overwrite coords, so we can add the freezing point (with uhrho_et and vhrho_nt) without problems:
	yq = cc.querying.getvar(exp,'yq',session,n=1)
	yq = yq.sel(yq=lat_range)
	#Commented below as I'm not quite sure it needs to be done
	#vhrho_nt.coords['yh'] = yq.values
	#vhrho_nt = vhrho_nt.rename(({'yh':'yq'}))
	#uhrho_et.coords['xh'] = xq.values
	# uhrho_et = uhrho_et.rename(({'xh':'xq'}))
	
	# interpolating the Salinity and volume into the u and v grid
	SalU_daily = Sal_daily.interp(xh=uhrho_et.xq)
	DztU = Dzt.interp(xh=uhrho_et.xq).sel(yh=lat_range)
	Salt_trans_zonal = (rho_0*uhrho_et*SalU_daily*DztU).rename({'z_l_sub01':'zl'})
	del DztU
	DztV = Dzt.interp(yh=vhrho_nt.yq)
	SalV_daily = Sal_daily.interp(yh=vhrho_nt.yq)
	del Sal_daily,Dzt
	Salt_trans_meridional = (rho_0*vhrho_nt*SalV_daily*DztV).rename({'z_l_sub01':'zl'})
	del vhrho_nt,uhrho_et,DztV
	print('Lets fitst do some dimension checking')
	print('SalU_daily = ')
	print(SalU_daily)
	print('SalV_daily = ')
	print(SalV_daily)
	#print('DztU = ')
	#print(DztU)
	#print('DztV = ')
	#print(DztV)
	
	
	xmax=int(np.size(Salt_trans_zonal.xq))
	ymax=int(np.size(Salt_trans_meridional.yq))
	
	Salt_trans_zonal=Salt_trans_zonal.isel(xq=slice(1,xmax)).chunk(chunks={"xq":"200gb","yh":"200gb"})
	Salt_trans_meridional=Salt_trans_meridional.isel(yq=slice(1,ymax)).chunk(chunks={"xh":"200gb","yq":"200gb"})
	
	#
	print('number of days in the current month = ' +str(ndays_month))
	print('Salt_trans_zonal = ')
	print(Salt_trans_zonal)
	print('Salt_trans_meridional = ')
	print(Salt_trans_meridional)


importing daily mass transports and salinities
0.3.0
0.3.0
0.3.0
Lets fitst do some dimension checking
SalU_daily = 
<xarray.DataArray 'so' (time: 31, z_l_sub01: 51, yh: 1019, xq: 7201)>
dask.array<chunked_aware_interpnd, shape=(31, 51, 1019, 7201), dtype=float32, chunksize=(1, 38, 1019, 7201), chunktype=numpy.ndarray>
Coordinates:
  * yh         (yh) float64 -81.1 -81.08 -81.06 -81.03 ... -59.07 -59.05 -59.02
  * z_l_sub01  (z_l_sub01) float64 0.5413 1.681 2.94 ... 1.333e+03 1.453e+03
  * time       (time) object 2000-01-01 12:00:00 ... 2000-01-31 12:00:00
    xh         (xq) float64 -280.0 -279.9 -279.9 -279.8 ... 79.9 79.95 80.0
  * xq         (xq) float64 -280.0 -279.9 -279.9 -279.8 ... 79.9 79.95 80.0
Attributes:
    units:          psu
    long_name:      Sea Water Salinity
    cell_methods:   area:mean z_l:mean yh:mean xh:mean time: mean
    cell_measures:  volume: volcello area: areacello
    time_avg_info:  average_T1,average_T2,average_DT
    standard_name:  sea_water_salinit

In [7]:
Salt_trans_zonal.load()

2023-12-28 14:10:21,484 - distributed.worker - ERROR - failed during get data with tcp://127.0.0.1:44367 -> tcp://127.0.0.1:44105
Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-23.10/lib/python3.10/site-packages/tornado/iostream.py", line 861, in _read_to_buffer
    bytes_read = self.read_from_fd(buf)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-23.10/lib/python3.10/site-packages/tornado/iostream.py", line 1116, in read_from_fd
    return self.socket.recv_into(buf, len(buf))
TimeoutError: [Errno 110] Connection timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-23.10/lib/python3.10/site-packages/distributed/worker.py", line 1784, in get_data
    response = await comm.read(deserializers=serializers)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-23.10/lib/python3.10/site-packages/distributed/comm/

KilledWorker: Attempted to run task ('rechunk-split-rechunk-merge-66ae1f1aa274af46598cde41d820215c', 3, 2, 4, 16) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://127.0.0.1:44257. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

2023-12-28 14:22:28,144 - distributed.worker - ERROR - failed during get data with tcp://127.0.0.1:43975 -> tcp://127.0.0.1:42411
Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-23.10/lib/python3.10/site-packages/tornado/iostream.py", line 861, in _read_to_buffer
    bytes_read = self.read_from_fd(buf)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-23.10/lib/python3.10/site-packages/tornado/iostream.py", line 1116, in read_from_fd
    return self.socket.recv_into(buf, len(buf))
TimeoutError: [Errno 110] Connection timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-23.10/lib/python3.10/site-packages/distributed/worker.py", line 1784, in get_data
    response = await comm.read(deserializers=serializers)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-23.10/lib/python3.10/site-packages/distributed/comm/

In [8]:
	#Salt_trans_meridional.load()